# Configuration

In [1]:
import os 

if True ^ os.getcwd().endswith('hte-and-targeting'):
    os.chdir('..')

In [2]:
import pandas as pd 
import numpy as np

In [3]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['text.usetex'] = False

In [4]:
from statsmodels.regression.linear_model import OLS

In [5]:
from core.variables import * 
from core.segment_targeting_estimators import *
from core.dgp import *
from core.segment_targeting_experiments import *
from core.visualization import *

# DGP

$$
Y_i = \beta_0 + \beta_1'X_i + \beta_2 T_i + \beta_3' X_i T_i + \epsilon_i
$$

In [6]:
treatment_space = np.array([0, 1, 2, 3])

class LinearDemandDGP(object): 
    """ 
    Y = const + cov_map @ covariates + tau * treatment + interaction_map @ covariates * treatment + epsilon
    """
    def __init__(self, num_covariates: int, treatment_space: np.ndarray, seed: int = None) -> None:
        if seed is not None:
            np.random.seed(seed)

        # model parameters
        self.num_covariates = num_covariates
        self.treatment_space = treatment_space

        # draw coefficients
        self.const = np.random.normal(1, 1)
        self.tau = np.random.uniform(0, 1)
        self.cov_map = np.random.normal(0, 1, size=(num_covariates, ))  # shape = (num_covariates, )
        self.interaction_map = np.random.normal(0, 1, size=(num_covariates, ))  # shape = (num_covariates, )

    def sample(self, sample_size, seed: int = None) -> pd.DataFrame:
        if seed is not None:
            np.random.seed(seed)

        # generate individuals for training data
        covariates = self.sample_individuals(sample_size, seed=seed)  # shape = (sample_size, num_covariates)

        # randomly assign treatment
        treatment = np.random.choice(self.treatment_space, size=sample_size)  # shape = (sample_size, )
        
        # calculate outcome
        epsilon = np.random.normal(0, 1, size=sample_size)  # shape = (sample_size, )
        Y = self.const + covariates @ self.cov_map + self.tau * treatment + (covariates @ self.interaction_map) * treatment + epsilon

        # return 
        return_dict = {'outcome': Y, 'treatment': treatment}
        return_dict.update({f'cov_{i}': covariates[:, i] for i in range(self.num_covariates)})
        return pd.DataFrame(return_dict)
    
    def sample_individuals(self, sample_size, seed: int = None) -> np.ndarray:
        if seed is not None:
            np.random.seed(seed)

        return np.random.normal(0, 1, size=(sample_size, self.num_covariates))

In [7]:
linear_demand_dgp = LinearDemandDGP(
    num_covariates=10, 
    treatment_space=np.array([0, 1, 2, 3]), 
    seed=0
)

In [8]:
train_df = linear_demand_dgp.sample(sample_size=1000, seed=0)
train_df

,outcome,treatment,cov_0,cov_1,cov_2,cov_3,cov_4,cov_5,cov_6,cov_7,cov_8,cov_9
0,2.574603,0,1.764052,0.400157,0.978738,2.240893,1.867558,-0.977278,0.950088,-0.151357,-0.103219,0.410599
1,-0.123465,0,0.144044,1.454274,0.761038,0.121675,0.443863,0.333674,1.494079,-0.205158,0.313068,-0.854096
2,6.050862,0,-2.552990,0.653619,0.864436,-0.742165,2.269755,-1.454366,0.045759,-0.187184,1.532779,1.469359
3,4.327832,1,0.154947,0.378163,-0.887786,-1.980796,-0.347912,0.156349,1.230291,1.202380,-0.387327,-0.302303
4,-11.553577,3,-1.048553,-1.420018,-1.706270,1.950775,-0.509652,-0.438074,-1.252795,0.777490,-1.613898,-0.212740
...,...,...,...,...,...,...,...,...,...,...,...,...
995,3.600220,2,-0.622801,1.240477,0.170446,1.986898,0.111301,1.209655,-0.457330,0.923183,1.325519,-0.953166
996,-2.696689,0,-0.678130,0.334100,-0.754928,-0.388383,-0.626576,-1.306421,-0.038496,0.143567,-0.789074,-0.098504
997,-4.366878,2,-1.452866,-0.947866,-0.869579,-0.305517,-0.605147,0.224821,-1.038472,0.771819,-0.480009,1.834745
998,-0.001486,1,-0.108765,0.454475,0.098709,-0.458305,-0.857044,-0.774403,0.708787,0.018474,-0.104081,-0.234858


# Demand Models

In [9]:
class LinearModel(object):
    def __init__(self, num_covariates: int, treatment_space: np.ndarray) -> None:
        self.num_covariates = num_covariates
        self.treatment_space = treatment_space

        # placeholder
        self.ols = None 

    def fit(self, data: pd.DataFrame) -> None:
        outcome_arr = data['outcome'].values.reshape(-1, 1)  # shape = (sample_size, 1)
        treatment_arr = data['treatment'].values.reshape(-1, 1)  # shape = (sample_size, 1)
        covariate_arr = data[[f'cov_{i}' for i in range(self.num_covariates)]].values  # shape = (sample_size, num_covariates)
        interaction_arr = covariate_arr * treatment_arr  # shape = (sample_size, num_covariates)

        exog_df = pd.DataFrame(
            np.concatenate((np.ones_like(treatment_arr), treatment_arr, covariate_arr, interaction_arr), axis=1), 
            columns=['const', 'tau'] + [f'cov_{i}' for i in range(self.num_covariates)] + [f'interaction_{i}' for i in range(self.num_covariates)]
        )

        self.ols = OLS(outcome_arr, exog_df).fit()

    def predict(self, data: pd.DataFrame) -> np.ndarray:
        treatment_arr = data['treatment'].values.reshape(-1, 1)
        covariate_arr = data[[f'cov_{i}' for i in range(self.num_covariates)]].values
        interaction_arr = covariate_arr * treatment_arr
        return self.ols.predict(
            np.concatenate((
                np.ones_like(treatment_arr), 
                treatment_arr, 
                covariate_arr, 
                interaction_arr
            ), axis=1)
        )

    def summary(self):
        return self.ols.summary()

In [10]:
linear_model = LinearModel(
    num_covariates=10, 
    treatment_space=np.array([0, 1, 2, 3])
)

linear_model.fit(train_df)
linear_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.980
Model:                            OLS   Adj. R-squared:                  0.979
Method:                 Least Squares   F-statistic:                     2234.
Date:                Sat, 07 Sep 2024   Prob (F-statistic):               0.00
Time:                        15:28:25   Log-Likelihood:                -1417.4
No. Observations:                1000   AIC:                             2879.
Df Residuals:                     978   BIC:                             2987.
Df Model:                          21                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             2.6810      0.054     50.025      0.000       2.576       2.786
tau               0.6564      0.029     22.934      0.000       0.600       0.713
cov_0             0.3830      0.052      7.311      0.000       0.280       0.486
cov_1            -2.2788      0.056    -40.579      0.000      -2.389      -2.169
cov_2             1.2783      0.054     23.786      0.000       1.173       1.384
cov_3            -0.8962      0.056    -15.960      0.000      -1.006      -0.786
cov_4             1.9819      0.054     36.944      0.000       1.877       2.087
cov_5             1.2451      0.056     22.067      0.000       1.134       1.356
cov_6            -0.4444      0.056     -7.895      0.000      -0.555      -0.334
cov_7             2.5831      0.055     46.903      0.000       2.475       2.691
cov_8             1.1224      0.054     20.797      0.000       1.016       1.228
cov_9             0.4336      0.054      7.962      0.000       0.327       0.541
interaction_0     0.5835      0.028     20.703      0.000       0.528       0.639
interaction_1    -0.1886      0.030     -6.321      0.000      -0.247      -0.130
interaction_2     1.4273      0.028     50.367      0.000       1.372       1.483
interaction_3    -0.3952      0.030    -13.149      0.000      -0.454      -0.336
interaction_4     0.2600      0.029      8.920      0.000       0.203       0.317
interaction_5    -0.9474      0.030    -31.852      0.000      -1.006      -0.889
interaction_6     0.3628      0.031     11.718      0.000       0.302       0.424
interaction_7     0.0181      0.028      0.641      0.522      -0.037       0.074
interaction_8     0.6748      0.029     23.278      0.000       0.618       0.732
interaction_9    -1.5431      0.030    -51.579      0.000      -1.602      -1.484
==============================================================================
Omnibus:                        3.300   Durbin-Watson:                   1.865
Prob(Omnibus):                  0.192   Jarque-Bera (JB):                3.194
Skew:                           0.108   Prob(JB):                        0.203
Kurtosis:                       3.172   Cond. No.                         5.05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [11]:
pd.DataFrame({
    'y_true': train_df['outcome'],
    'y_pred': linear_model.predict(train_df)
})

,y_true,y_pred
0,2.574603,3.420962
1,-0.123465,0.368073
2,6.050862,6.524907
3,4.327832,4.643208
4,-11.553577,-10.746038
...,...,...
995,3.600220,4.673875
996,-2.696689,-2.365923
997,-4.366878,-4.802708
998,-0.001486,0.881512


In [12]:
class PotentialOutcomeModel(object):
    def __init__(self, num_covariates: int, treatment_space: np.ndarray) -> None:
        self.num_covariates = num_covariates
        self.treatment_space = treatment_space

        # placeholder
        self.lift_dict = {t: np.nan for t in self.treatment_space if t != 0}

    def fit(self, data):
        for t in self.treatment_space:
            if t == 0:
                continue
            mask = (data['treatment'] == t) | (data['treatment'] == 0)
            self.lift_dict[t] = self.difference_in_mean(
                treatment_val=t, 
                treatments=data['treatment'].values[mask], 
                outcomes=data['outcome'].values[mask]
            )
        
    @staticmethod
    def difference_in_mean(treatment_val: int, treatments: np.ndarray, outcomes: np.ndarray) -> float:
        return outcomes[treatments == treatment_val].mean() - outcomes[treatments == 0].mean()

In [13]:
po_model = PotentialOutcomeModel(
    num_covariates=10, 
    treatment_space=np.array([0, 1, 2, 3])
)
po_model.fit(train_df)

In [14]:
po_model.lift_dict

{1: 0.3766364340123296, 2: 0.08973553894112296, 3: 1.2906542640663279}

In [51]:
# TODO: Change covariates X_i to be consumer segments